# SupplySight – exploratory data analysis

**All data is synthetic.** Run `python run.py data` first. This notebook is for exploration only; the pipeline never depends on it.

In [ ]:
import sys

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import pandas as pd

from supplysight.data import load_source_tables, po_items, weekly_consumption

T = load_source_tables()
{k: len(v) for k, v in T.items()}

## Seasonality of finished-goods production
Sunscreens peak in summer; lipsticks and creams before Diwali and the wedding season.

In [ ]:
af = T["afko"].merge(T["mara"][["matnr", "matkl"]], left_on="plnbez", right_on="matnr")
af["month"] = pd.to_datetime(af.gstrp).dt.to_period("M").dt.to_timestamp()
g = af.groupby(["month", "matkl"]).gamng.sum().unstack()
(g[["FG-SUN", "FG-LIP", "FG-CREAM", "FG-HAIR"]] / g[["FG-SUN", "FG-LIP", "FG-CREAM", "FG-HAIR"]].mean()).iloc[1:].plot(
    figsize=(11, 4), title="Production index by group"
);

## Raw-material consumption is derived from production × BOM

In [ ]:
wc = weekly_consumption(T["mseg"])
top = wc.groupby("matnr").qty.sum().nlargest(4).index
wc[wc.matnr.isin(top)].pivot_table(index="week", columns="matnr", values="qty", aggfunc="sum").plot(
    figsize=(11, 4), title="Weekly consumption, top-4 materials"
);

## Price distributions and injected anomalies

In [ ]:
items = po_items(T).merge(T["anomaly_labels"], on=["ebeln", "ebelp"], how="left")
med = items[items.anomaly_type.isna()].groupby("matnr").netpr.median()
items["price_ratio"] = items.netpr / items.matnr.map(med)

normal = items[items.anomaly_type.isna()].price_ratio.clip(0.5, 3)
spikes = items[items.anomaly_type == "price_spike"].price_ratio.clip(0.5, 3)
ax = normal.plot.hist(bins=60, alpha=0.6, density=True, label="normal", figsize=(8, 3.5))
spikes.plot.hist(bins=30, alpha=0.6, density=True, ax=ax, label="price spike")
ax.legend()
items.anomaly_type.value_counts()

## Lead times and PO timing

In [ ]:
items["lead_time"] = (items.eindt - items.bedat).dt.days
print(items.groupby(items.anomaly_type.fillna("normal")).lead_time.describe().round(1))

hours = items.created_at.dt.hour.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(hours.index, hours.values)
ax.set_title("PO creation hour")